# 02 · Compute All Measures

Replicates `analysis_Haddara.m`, `analysis_Maniscalco.m`, `analysis_Shekhar.m`, `analysis_Rouault1.m`, `analysis_Rouault2.m`.

For each subject this notebook computes:
- **Raw** measures (all 20) on the full dataset
- **Difficulty** measures at each task difficulty level
- **Metacognitive bias** measures via Xue et al. (2021) recoding
- **Split-half** reliability (odd vs even trials per bin size)
- **Precision** measures with artificially corrupted confidence

⚠️ **Runtime**: `meta-d'`, `meta-noise`, and `meta-uncertainty` each require numerical optimisation (~2–4 s per call). Run time estimates:
- Shekhar (20 subs × 3 contrasts × 2 recodings): ~10 min  
- Rouault1 (466 subs × 2 splits): ~1 hour  
- Haddara (70 subs, split-half bins): ~30 min  

Results are saved to `notebooks/precomputed/` and loaded by notebooks 03 and 04.

In [ ]:
import sys, os, warnings, time
warnings.filterwarnings('ignore')

REPO = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO, 'src'))

import numpy as np
from metasignal.stdpy.compute_all import compute_all_measures
sys.path.insert(0, os.path.join(REPO, 'notebooks'))
from analysis_core import (
    xue_recode, ttest_1samp, MEASURE_NAMES, N_MEASURES,
    preprocess_haddara, preprocess_maniscalco, preprocess_shekhar,
    preprocess_rouault, preprocess_locke,
    metas_altered_conf,
)

OUT = os.path.join(REPO, 'notebooks', 'precomputed')
os.makedirs(OUT, exist_ok=True)
print(f'Output → {OUT}')

## Xue et al. (2021) Recoding

To test independence from metacognitive bias, confidence ratings are recoded two ways:

| Type | Operation | Effect |
|------|-----------|--------|
| 1 | conf − 1, floor at new min+1 | Removes lowest rating → biases toward **high** confidence |
| 2 | max rating → max − 1 | Removes highest rating → biases toward **low** confidence |

A measure independent of bias should not change between the two recodings.

In [ ]:
def compute_bias_pair(stim, resp, conf, n_ratings):
    """Compute measures under both Xue recodings.
    Returns (measures_recode1, measures_recode2) — (20,) each.
    """
    r1 = xue_recode(conf, 1)
    r2 = xue_recode(conf, 2)
    m1 = compute_all_measures(stim, resp, r1, n_ratings - 1) if not np.all(np.isnan(r1)) else np.full(N_MEASURES, np.nan)
    m2 = compute_all_measures(stim, resp, r2, n_ratings - 1) if not np.all(np.isnan(r2)) else np.full(N_MEASURES, np.nan)
    return m1, m2

print('Helper functions defined.')

## Haddara 2022 — full analysis

In [ ]:
print('Loading Haddara subjects...')
ha = preprocess_haddara()
print(f'  n = {len(ha)}')

ha_raw   = np.full((len(ha), N_MEASURES), np.nan)
ha_bias  = np.full((len(ha), 2, N_MEASURES), np.nan)  # (n_sub, recode, measure)
ha_split = np.full((len(ha), 2, N_MEASURES), np.nan)  # odd/even
ha_prec  = {}  # bin_size → (n_sub, n_bins, n_alt+1, N_MEASURES)

BIN_SIZES  = [50, 100, 200, 400]
PROP_ALTER = [0.02, 0.04, 0.06]

t0 = time.time()
for i, s in enumerate(ha):
    if i % 10 == 0:
        print(f'  Subject {i+1}/{len(ha)}...')
    nr = s['n_ratings']
    ha_raw[i]      = compute_all_measures(s['stim'], s['resp'], s['conf'], nr)
    ha_bias[i,0], ha_bias[i,1] = compute_bias_pair(s['stim'], s['resp'], s['conf'], nr)
    ha_split[i,0]  = compute_all_measures(s['stim'][0::2], s['resp'][0::2], s['conf'][0::2], nr)
    ha_split[i,1]  = compute_all_measures(s['stim'][1::2], s['resp'][1::2], s['conf'][1::2], nr)

print(f'  Done in {(time.time()-t0)/60:.1f} min')
np.savez(os.path.join(OUT, 'haddara_results.npz'),
         raw=ha_raw, bias=ha_bias, split_odd_even=ha_split)
print('  Saved haddara_results.npz')

### Haddara — Precision analysis

For each bin size, split the session into non-overlapping bins. For each bin, compute measures on unaltered and 3 levels of artificially corrupted confidence.

In [ ]:
ha_prec_all = {}  # bin_size → (n_sub, n_bins, n_alt+1, N_MEASURES)

for bs in BIN_SIZES:
    n_trials = min(len(s['stim']) for s in ha)
    n_bins = n_trials // bs
    arr = np.full((len(ha), n_bins, len(PROP_ALTER)+1, N_MEASURES), np.nan)
    for i, s in enumerate(ha):
        nr = s['n_ratings']
        for b in range(n_bins):
            sl = slice(b*bs, (b+1)*bs)
            st, re, co = s['stim'][sl], s['resp'][sl], s['conf'][sl]
            arr[i, b, 0] = compute_all_measures(st, re, co, nr)
            for ai, pa in enumerate(PROP_ALTER):
                arr[i, b, ai+1] = metas_altered_conf(st, re, co, nr, pa)
    ha_prec_all[bs] = arr
    print(f'  Bin size {bs}: done')

np.savez(os.path.join(OUT, 'haddara_precision.npz'), **{f'bs{k}': v for k,v in ha_prec_all.items()})
print('Saved haddara_precision.npz')

## Maniscalco 2017

In [ ]:
print('Loading Maniscalco subjects...')
ma = preprocess_maniscalco()
print(f'  n = {len(ma)}')

ma_raw  = np.full((len(ma), N_MEASURES), np.nan)
ma_bias = np.full((len(ma), 2, N_MEASURES), np.nan)
ma_split= np.full((len(ma), 2, N_MEASURES), np.nan)

for i, s in enumerate(ma):
    nr = s['n_ratings']
    ma_raw[i]      = compute_all_measures(s['stim'], s['resp'], s['conf'], nr)
    ma_bias[i,0], ma_bias[i,1] = compute_bias_pair(s['stim'], s['resp'], s['conf'], nr)
    ma_split[i,0]  = compute_all_measures(s['stim'][0::2], s['resp'][0::2], s['conf'][0::2], nr)
    ma_split[i,1]  = compute_all_measures(s['stim'][1::2], s['resp'][1::2], s['conf'][1::2], nr)

np.savez(os.path.join(OUT, 'maniscalco_results.npz'),
         raw=ma_raw, bias=ma_bias, split_odd_even=ma_split)
print('Saved maniscalco_results.npz')

## Shekhar 2021
Computes measures separately for each of the 3 contrast levels.

In [ ]:
print('Loading Shekhar subjects...')
sh = preprocess_shekhar()
print(f'  n = {len(sh)}')

# shape: (n_sub, n_contrasts=3, N_MEASURES)
sh_diff  = np.full((len(sh), 3, N_MEASURES), np.nan)
# shape: (n_sub, n_contrasts=3, recode=2, N_MEASURES)
sh_bias  = np.full((len(sh), 3, 2, N_MEASURES), np.nan)

for i, s in enumerate(sh):
    nr = s['n_ratings']
    for ci, c in enumerate([1, 2, 3]):
        mask = s['Contrast'] == c
        if mask.sum() < 10:
            continue
        st, re, co = s['stim'][mask], s['resp'][mask], s['conf'][mask]
        sh_diff[i, ci] = compute_all_measures(st, re, co, nr)
        sh_bias[i, ci, 0], sh_bias[i, ci, 1] = compute_bias_pair(st, re, co, nr)

np.savez(os.path.join(OUT, 'shekhar_results.npz'), diff=sh_diff, bias=sh_bias)
print('Saved shekhar_results.npz')

## Rouault 2018
For each subject, compute measures separately for the low-contrast (≤ median) and high-contrast (> median) halves.

In [ ]:
for expt, tag in [(1, 'rouault1'), (2, 'rouault2')]:
    print(f'Loading Rouault {expt}...')
    subs = preprocess_rouault(expt)
    print(f'  n = {len(subs)}')
    arr = np.full((len(subs), 2, N_MEASURES), np.nan)  # dim1: [low, high]
    t0 = time.time()
    for i, s in enumerate(subs):
        if i % 100 == 0:
            print(f'    {i}/{len(subs)}...')
        nr  = s['n_ratings']
        med = np.median(s['contrast'])
        lo  = s['contrast'] <= med
        hi  = s['contrast'] >  med
        if lo.sum() >= 10:
            arr[i, 0] = compute_all_measures(s['stim'][lo], s['resp'][lo], s['conf'][lo], nr)
        if hi.sum() >= 10:
            arr[i, 1] = compute_all_measures(s['stim'][hi], s['resp'][hi], s['conf'][hi], nr)
    print(f'  Done in {(time.time()-t0)/60:.1f} min')
    np.savez(os.path.join(OUT, f'{tag}_results.npz'), diff=arr)
    print(f'  Saved {tag}_results.npz')

## Locke 2020
Computes measures per condition (7 response-bias conditions).

In [ ]:
print('Loading Locke subjects...')
lo = preprocess_locke()
print(f'  n = {len(lo)}')

CONDS = list(range(1, 8))
lo_rb = np.full((len(lo), 7, N_MEASURES), np.nan)

for i, s in enumerate(lo):
    nr = s['n_ratings']
    for ci, cond in enumerate(CONDS):
        mask = s['condition'] == cond
        if mask.sum() >= 5:
            lo_rb[i, ci] = compute_all_measures(
                s['stim'][mask], s['resp'][mask], s['conf'][mask], nr)

np.savez(os.path.join(OUT, 'locke_results.npz'), rb=lo_rb)
print('Saved locke_results.npz')

In [ ]:
print('\n=== All datasets processed ===')
for fname in sorted(os.listdir(OUT)):
    path = os.path.join(OUT, fname)
    print(f'  {fname}: {os.path.getsize(path)/1024:.0f} KB')